# MVP-1 — Cierre: Encoder Visual Final (A2)

**Decisión:** A2 (MAE + ranking escalar + consistency, solo 10x) es el encoder visual
final para MVP-1, seleccionado tras 7 variantes experimentales.

**Este notebook:**
1. Re-entrena A2 como modelo definitivo
2. Genera todos los diagnósticos (batch probe incremental, fusion-readiness)
3. Documenta la serie experimental completa como evidencia
4. Documenta limitaciones explícitamente
5. Exporta modelo + embeddings para MVP-2

---

## SECCIÓN 0 — CONFIG

In [1]:
import os

MANIFEST_PATH = "/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_mvp1_finetune_20260319_153838.csv"
CSV_CENTRAL   = "/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv"
IMAGE_ROOT    = "/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images"
OUTPUT_DIR    = "/Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final"

COL_IMG_PATH      = "img_path"
COL_PDL_NORM      = "pdl_norm"
COL_PDL_RAW       = "pdl"
COL_FOLD          = "fold"
COL_CELL_LINE     = "cell_line"
COL_STUDY_PART    = "study_part"
COL_MAGNIFICATION = "magnification"
COL_SAMPLE_ID     = "sample_id"
COL_TELOMERE      = "telomere_length"
COL_MTDNA         = "mtdna_cn"

# Hiperparámetros A2 (los mismos del run ganador)
BATCH_SIZE    = 16
EPOCHS        = 30
PATIENCE      = 7
LR_HEAD       = 1e-3
WEIGHT_DECAY  = 1e-4
EMBEDDING_DIM = 256
IMG_SIZE      = 224
SEED          = 42
LAMBDA_PDL    = 1.0
LAMBDA_RANK   = 0.3
LAMBDA_CONS   = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config cargada. Output → {OUTPUT_DIR}")

✅ Config cargada. Output → /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final


## SECCIÓN 1 — IMPORTS

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

from PIL import Image

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🖥  Device: Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🖥  Device: CUDA — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("🖥  Device: CPU")

🖥  Device: Apple MPS


## SECCIÓN 2 — CARGAR MANIFEST (solo 10x)

In [3]:
df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest cargado: {df.shape[0]} filas")

if not os.path.isabs(df[COL_IMG_PATH].iloc[0]):
    df["_img_abs"] = df[COL_IMG_PATH].apply(lambda p: os.path.join(IMAGE_ROOT, p))
else:
    df["_img_abs"] = df[COL_IMG_PATH]

exists_mask = df["_img_abs"].apply(os.path.exists)
df = df[exists_mask].reset_index(drop=True)

n_antes = len(df)
df = df[df[COL_MAGNIFICATION].astype(str) == "10"].reset_index(drop=True)
print(f"🔬 Filtro 10x: {n_antes} → {len(df)} imágenes")

# Codificar cell_line para ranking loss
cell_lines = sorted(df[COL_CELL_LINE].unique())
cl_to_idx = {cl: i for i, cl in enumerate(cell_lines)}
df["_cl_idx"] = df[COL_CELL_LINE].map(cl_to_idx)

print(f"\n📊 Resumen:")
print(f"   Imágenes: {len(df)}")
print(f"   Cell lines: {cell_lines}")
print(f"   Folds: {df[COL_FOLD].value_counts().sort_index().to_dict()}")
print(f"   Study parts: {sorted(df[COL_STUDY_PART].unique())}")
print(f"   PDL norm: [{df[COL_PDL_NORM].min():.3f}, {df[COL_PDL_NORM].max():.3f}]")

# Stats por fold
print(f"\n📊 Detalle por fold:")
for fold in sorted(df[COL_FOLD].unique()):
    sub = df[df[COL_FOLD] == fold]
    cls = sorted(sub[COL_CELL_LINE].unique())
    phases = sorted(sub[COL_STUDY_PART].unique())
    print(f"   Fold {fold}: {len(sub)} imgs | cells: {cls} | phases: {phases}")

Manifest cargado: 763 filas
🔬 Filtro 10x: 763 → 393 imágenes

📊 Resumen:
   Imágenes: 393
   Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2']
   Folds: {0.0: 127, 1.0: 217, 2.0: 49}
   Study parts: [1, 2, 3, 5]
   PDL norm: [0.047, 1.000]

📊 Detalle por fold:
   Fold 0.0: 127 imgs | cells: ['hFB12'] | phases: [2, 3, 5]
   Fold 1.0: 217 imgs | cells: ['hFB13', 'hFB14'] | phases: [2, 3, 5]
   Fold 2.0: 49 imgs | cells: ['hFB1', 'hFB11', 'hFB2'] | phases: [1, 3]


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_71116/3937894777.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_img_abs"] = df[COL_IMG_PATH]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_71116/3937894777.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_cl_idx"] = df[COL_CELL_LINE].map(cl_to_idx)


## SECCIÓN 3 — DATASET, MODELO, PÉRDIDAS

In [4]:
# --- Dataset ---
class A2Dataset(Dataset):
    def __init__(self, dataframe, transform, transform_aug=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.transform_aug = transform_aug or transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["_img_abs"])
        if img.mode != "RGB":
            img = img.convert("RGB")
        img1 = self.transform(img)
        img2 = self.transform_aug(img)
        pdl = torch.tensor(row[COL_PDL_NORM], dtype=torch.float32)
        cl_idx = torch.tensor(row["_cl_idx"], dtype=torch.long)
        return img1, img2, pdl, cl_idx


# --- Transforms ---
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# --- Modelo ---
class PDLEncoderA2(nn.Module):
    """ResNet-34 (frozen) → embedding 256-dim → PDL prediction."""

    def __init__(self, embedding_dim=256, freeze_backbone=True):
        super().__init__()
        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        backbone_out = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.embed_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.pdl_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embed_head(features)
        pdl_hat = self.pdl_head(embedding).squeeze(-1)
        return pdl_hat, embedding


# --- Pérdidas ---
def ranking_loss_scalar(pdl_hat, pdl_true, cl_idx):
    """Ranking intra cell_line usando output escalar del pdl_head."""
    loss = torch.tensor(0.0, device=pdl_hat.device)
    n_pairs = 0
    for cl in cl_idx.unique():
        mask = cl_idx == cl
        if mask.sum() < 2:
            continue
        scores = pdl_hat[mask]
        targets = pdl_true[mask]
        n = scores.shape[0]
        for i in range(n):
            for j in range(i + 1, n):
                diff = abs(targets[i] - targets[j])
                if diff < 0.05:
                    continue
                if targets[i] < targets[j]:
                    loss += torch.relu(diff * 0.3 - (scores[j] - scores[i]))
                else:
                    loss += torch.relu(diff * 0.3 - (scores[i] - scores[j]))
                n_pairs += 1
    return loss / max(n_pairs, 1)


def consistency_loss(emb1, emb2):
    return torch.mean((emb1 - emb2) ** 2)


print("✅ Dataset, modelo y pérdidas definidos")

✅ Dataset, modelo y pérdidas definidos


## SECCIÓN 4 — ENTRENAMIENTO A2 DEFINITIVO (3-fold CV)

In [5]:
criterion_pdl = nn.L1Loss()
folds = sorted(df[COL_FOLD].unique())

print("=" * 60)
print("  ENTRENAMIENTO A2 FINAL — 3-FOLD CV")
print("=" * 60)

results_per_fold = {}
all_embeddings = []

for val_fold in folds:
    print(f"\n{'─'*55}")
    print(f"  Fold {val_fold}")
    print(f"{'─'*55}")

    df_train = df[df[COL_FOLD] != val_fold]
    df_val = df[df[COL_FOLD] == val_fold]
    print(f"  Train: {len(df_train)} | Val: {len(df_val)} | "
          f"Val cells: {sorted(df_val[COL_CELL_LINE].unique())}")

    ds_train = A2Dataset(df_train, train_transform, train_transform_aug)
    ds_val = A2Dataset(df_val, val_transform, val_transform)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    trivial_mae = np.mean(np.abs(
        df_val[COL_PDL_NORM].values - df_train[COL_PDL_NORM].mean()))

    model = PDLEncoderA2(EMBEDDING_DIM, freeze_backbone=True).to(DEVICE)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

    best_mae = float("inf")
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        for img1, img2, pdl, cl_idx in dl_train:
            img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
            pdl, cl_idx = pdl.to(DEVICE), cl_idx.to(DEVICE)

            optimizer.zero_grad()
            pdl_hat, emb1 = model(img1)

            loss = LAMBDA_PDL * criterion_pdl(pdl_hat, pdl)
            loss += LAMBDA_RANK * ranking_loss_scalar(pdl_hat, pdl, cl_idx)

            _, emb2 = model(img2)
            loss += LAMBDA_CONS * consistency_loss(emb1, emb2)

            loss.backward()
            optimizer.step()

        # Eval
        model.eval()
        val_pdl, val_hat, val_emb = [], [], []
        with torch.no_grad():
            for img1, img2, pdl, cl_idx in dl_val:
                img1 = img1.to(DEVICE)
                pdl_hat, emb = model(img1)
                val_pdl.extend(pdl.numpy())
                val_hat.extend(pdl_hat.cpu().numpy())
                val_emb.append(emb.cpu().numpy())

        val_pdl_arr = np.array(val_pdl)
        val_hat_arr = np.array(val_hat)
        mae = np.mean(np.abs(val_pdl_arr - val_hat_arr))
        rho, p_val = spearmanr(val_pdl_arr, val_hat_arr)

        improved = ""
        if mae < best_mae:
            best_mae = mae
            patience_counter = 0
            improved = "✓"
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_emb = np.vstack(val_emb)
            best_hat = val_hat_arr.copy()
            best_rho = rho
            best_p = p_val
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or improved or epoch == 0:
            print(f"  Ep {epoch+1:2d} | MAE={mae:.4f} | ρ={rho:.3f} {improved}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping ep {epoch+1}")
            break

    # Guardar mejor modelo
    torch.save(best_state, os.path.join(OUTPUT_DIR, f"A2_final_fold{val_fold}.pt"))

    ss_res = np.sum((val_pdl_arr - best_hat) ** 2)
    ss_tot = np.sum((val_pdl_arr - val_pdl_arr.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    improvement = (1 - best_mae / trivial_mae) * 100

    # Embeddings
    df_emb = df_val.copy().reset_index(drop=True)
    df_emb["y_pred"] = best_hat
    emb_df = pd.DataFrame(best_emb, columns=[f"emb_{i}" for i in range(best_emb.shape[1])])
    df_emb = pd.concat([df_emb, emb_df], axis=1)
    all_embeddings.append(df_emb)

    results_per_fold[val_fold] = {
        "n_val": len(df_val),
        "cell_lines_val": sorted(df_val[COL_CELL_LINE].unique()),
        "trivial_mae": float(trivial_mae),
        "model_mae": float(best_mae),
        "improvement_pct": float(improvement),
        "spearman": float(best_rho),
        "spearman_p": float(best_p),
        "r2": float(r2),
    }

    print(f"  📊 MAE={best_mae:.4f} (mejora {improvement:+.1f}%) | "
          f"ρ={best_rho:.3f} | R²={r2:.3f}")

df_embeddings = pd.concat(all_embeddings, ignore_index=True)
print(f"\n✅ Entrenamiento completo. Embeddings: {df_embeddings.shape}")

  ENTRENAMIENTO A2 FINAL — 3-FOLD CV

───────────────────────────────────────────────────────
  Fold 0.0
───────────────────────────────────────────────────────
  Train: 266 | Val: 127 | Val cells: ['hFB12']
  Ep  1 | MAE=0.1659 | ρ=0.717 ✓
  Ep  2 | MAE=0.1454 | ρ=0.752 ✓
  Ep  5 | MAE=0.1505 | ρ=0.786 
  Ep  6 | MAE=0.1345 | ρ=0.781 ✓
  Ep  9 | MAE=0.1301 | ρ=0.807 ✓
  Ep 10 | MAE=0.1697 | ρ=0.798 
  Ep 14 | MAE=0.1261 | ρ=0.814 ✓
  Ep 15 | MAE=0.1425 | ρ=0.809 
  Ep 17 | MAE=0.1223 | ρ=0.809 ✓
  Ep 18 | MAE=0.1213 | ρ=0.809 ✓
  Ep 20 | MAE=0.1443 | ρ=0.800 
  Ep 25 | MAE=0.1373 | ρ=0.796 
  Early stopping ep 25
  📊 MAE=0.1213 (mejora +47.7%) | ρ=0.809 | R²=0.611

───────────────────────────────────────────────────────
  Fold 1.0
───────────────────────────────────────────────────────
  Train: 176 | Val: 217 | Val cells: ['hFB13', 'hFB14']
  Ep  1 | MAE=0.2040 | ρ=0.730 ✓
  Ep  2 | MAE=0.1572 | ρ=0.724 ✓
  Ep  3 | MAE=0.1484 | ρ=0.731 ✓
  Ep  5 | MAE=0.1384 | ρ=0.768 ✓
  Ep  7 | MAE=

## SECCIÓN 5 — RESULTADOS FINALES

In [6]:
maes = [r["model_mae"] for r in results_per_fold.values()]
spears = [r["spearman"] for r in results_per_fold.values()]
imps = [r["improvement_pct"] for r in results_per_fold.values()]

print("=" * 70)
print("  MVP-1 ENCODER FINAL — A2 (MAE + Ranking + Consistency, 10x)")
print("=" * 70)

for fold, r in results_per_fold.items():
    print(f"  Fold {fold}: MAE={r['model_mae']:.4f} | ρ={r['spearman']:.3f} | "
          f"R²={r['r2']:.3f} | mejora={r['improvement_pct']:+.1f}% | "
          f"cells={', '.join(r['cell_lines_val'])}")

print(f"\n  Promedio: MAE={np.mean(maes):.4f}±{np.std(maes):.4f} | "
      f"ρ={np.mean(spears):.3f}±{np.std(spears):.3f} | "
      f"mejora={np.mean(imps):+.1f}%")
print(f"  Worst fold ρ: {min(spears):.3f}")

s_ok = "✓" if np.mean(spears) >= 0.6 else "✗"
m_ok = "✓" if np.mean(imps) >= 25 else "✗"
w_ok = "✓" if min(spears) >= 0.6 else "✗"
print(f"\n  📋 Criterios PIDA:")
print(f"     Spearman ≥ 0.6:  {s_ok}  ({np.mean(spears):.3f})")
print(f"     Mejora ≥ 25%:    {m_ok}  ({np.mean(imps):+.1f}%)")
print(f"     Worst fold ≥ 0.6: {w_ok}  ({min(spears):.3f})")

  MVP-1 ENCODER FINAL — A2 (MAE + Ranking + Consistency, 10x)
  Fold 0.0: MAE=0.1213 | ρ=0.809 | R²=0.611 | mejora=+47.7% | cells=hFB12
  Fold 1.0: MAE=0.1178 | ρ=0.772 | R²=0.667 | mejora=+55.5% | cells=hFB13, hFB14
  Fold 2.0: MAE=0.1843 | ρ=0.619 | R²=0.380 | mejora=+29.0% | cells=hFB1, hFB11, hFB2

  Promedio: MAE=0.1411±0.0306 | ρ=0.733±0.082 | mejora=+44.1%
  Worst fold ρ: 0.619

  📋 Criterios PIDA:
     Spearman ≥ 0.6:  ✓  (0.733)
     Mejora ≥ 25%:    ✓  (+44.1%)
     Worst fold ≥ 0.6: ✓  (0.619)


## SECCIÓN 6 — BATCH PROBE INCREMENTAL

In [7]:
print("\n" + "=" * 70)
print("  BATCH PROBE INCREMENTAL")
print("=" * 70)

emb_cols = [c for c in df_embeddings.columns if c.startswith("emb_")]
X_emb = df_embeddings[emb_cols].values

# --- Study_part ---
y_sp = df_embeddings[COL_STUDY_PART].values
le_sp = LabelEncoder()
y_sp_enc = le_sp.fit_transform(y_sp)

X_struct = pd.get_dummies(
    df_embeddings[[COL_PDL_NORM, COL_CELL_LINE]],
    columns=[COL_CELL_LINE]).values.astype(float)

clf_a = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_a.fit(X_struct, y_sp_enc)
auc_struct = roc_auc_score(y_sp_enc, clf_a.predict_proba(X_struct),
                           multi_class="ovr", average="macro")

clf_b = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_b.fit(np.hstack([X_struct, X_emb]), y_sp_enc)
auc_full = roc_auc_score(y_sp_enc, clf_b.predict_proba(np.hstack([X_struct, X_emb])),
                         multi_class="ovr", average="macro")

clf_c = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_c.fit(X_emb, y_sp_enc)
auc_raw = roc_auc_score(y_sp_enc, clf_c.predict_proba(X_emb),
                        multi_class="ovr", average="macro")

delta_sp = auc_full - auc_struct

print(f"  Study_part (4 clases):")
print(f"    AUC solo z:              {auc_raw:.3f}")
print(f"    AUC PDL+cell_line:       {auc_struct:.3f}")
print(f"    AUC PDL+cell_line+z:     {auc_full:.3f}")
print(f"    ΔAUC incremental:        {delta_sp:+.3f}")
print(f"    Nota: PDL+cell_line ya predicen study_part con AUC={auc_struct:.3f}")
print(f"          porque study_part está acoplado con variables biológicas.")

# --- PDL bins ---
pdl_bins = pd.cut(df_embeddings[COL_PDL_NORM], bins=3, labels=[0, 1, 2])
clf_pdl = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
clf_pdl.fit(X_emb, pdl_bins)
auc_pdl = roc_auc_score(pdl_bins, clf_pdl.predict_proba(X_emb),
                        multi_class="ovr", average="macro")
print(f"\n  PDL bins (early/mid/late): AUC={auc_pdl:.3f}")

# --- Cell line ---
y_cl = df_embeddings[COL_CELL_LINE].values
le_cl = LabelEncoder()
y_cl_enc = le_cl.fit_transform(y_cl)
clf_cl = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_cl.fit(X_emb, y_cl_enc)
auc_cl = roc_auc_score(y_cl_enc, clf_cl.predict_proba(X_emb),
                       multi_class="ovr", average="macro")
print(f"  Cell line (6 clases):     AUC={auc_cl:.3f}")

probe_results = {
    "study_part": {"raw": float(auc_raw), "struct": float(auc_struct),
                   "full": float(auc_full), "delta": float(delta_sp)},
    "pdl_bins": float(auc_pdl),
    "cell_line": float(auc_cl),
}


  BATCH PROBE INCREMENTAL
  Study_part (4 clases):
    AUC solo z:              0.979
    AUC PDL+cell_line:       0.850
    AUC PDL+cell_line+z:     0.988
    ΔAUC incremental:        +0.139
    Nota: PDL+cell_line ya predicen study_part con AUC=0.850
          porque study_part está acoplado con variables biológicas.

  PDL bins (early/mid/late): AUC=0.936
  Cell line (6 clases):     AUC=0.973


## SECCIÓN 7 — FUSION-READINESS

In [8]:
print("\n" + "=" * 70)
print("  FUSION-READINESS — Correlación con biomarcadores")
print("=" * 70)

def partial_correlation(x, y, covariates):
    reg_x = LinearRegression().fit(covariates, x)
    res_x = x - reg_x.predict(covariates)
    reg_y = LinearRegression().fit(covariates, y)
    res_y = y - reg_y.predict(covariates)
    return spearmanr(res_x, res_y)

pca = PCA(n_components=1, random_state=SEED)
df_embeddings["emb_pc1"] = pca.fit_transform(X_emb)[:, 0]

bio_cols = [COL_TELOMERE, COL_MTDNA]
agg_dict = {"emb_pc1": "mean", COL_PDL_NORM: "first", COL_CELL_LINE: "first"}
for col in bio_cols:
    if col in df_embeddings.columns:
        agg_dict[col] = "first"

sub = df_embeddings.dropna(subset=[c for c in bio_cols if c in df_embeddings.columns])
agg = sub.groupby(COL_SAMPLE_ID).agg(agg_dict).reset_index()
cov = pd.get_dummies(agg[[COL_PDL_NORM, COL_CELL_LINE]],
                     columns=[COL_CELL_LINE]).values.astype(float)

# Correlaciones del dato (PDL vs biomarcadores)
print(f"\n  Correlaciones en el dato (referencia):")
mask_t = agg[COL_TELOMERE].notna()
mask_m = agg[COL_MTDNA].notna()
rho_pdl_t, _ = spearmanr(agg.loc[mask_t, COL_PDL_NORM], agg.loc[mask_t, COL_TELOMERE])
rho_pdl_m, _ = spearmanr(agg.loc[mask_m, COL_PDL_NORM], agg.loc[mask_m, COL_MTDNA])
print(f"    PDL vs telómero:  ρ={rho_pdl_t:+.3f}  (biología: más PDL → telómeros más cortos)")
print(f"    PDL vs mtDNA:     ρ={rho_pdl_m:+.3f}  (biología: más PDL → más copias mtDNA)")

fusion_results = {}
print(f"\n  Correlaciones z_img vs biomarcadores (agregado por sample_id, n={len(agg)}):")
for col in bio_cols:
    if col not in agg.columns:
        continue
    mask = agg[col].notna()
    n_valid = mask.sum()
    if n_valid < 15:
        continue

    x = agg.loc[mask, "emb_pc1"].values
    y = agg.loc[mask, col].values
    c = cov[mask.values]

    rho_s, p_s = spearmanr(x, y)
    rho_p, p_p = partial_correlation(x, y, c)

    sig_s = "*" if p_s < 0.05 else "ns"
    sig_p = "*" if p_p < 0.05 else "ns"

    print(f"    {col:25s} | simple ρ={rho_s:+.3f} ({sig_s}) | "
          f"parcial ρ={rho_p:+.3f} ({sig_p}, p={p_p:.3e}) | n={n_valid}")

    fusion_results[col] = {
        "simple_rho": float(rho_s), "simple_p": float(p_s),
        "partial_rho": float(rho_p), "partial_p": float(p_p),
        "n": int(n_valid),
    }

# PDL sanity
rho_pdl, p_pdl = spearmanr(agg["emb_pc1"], agg[COL_PDL_NORM])
print(f"    {'PDL (sanity check)':25s} | ρ={rho_pdl:+.3f} (p={p_pdl:.2e})")


  FUSION-READINESS — Correlación con biomarcadores

  Correlaciones en el dato (referencia):
    PDL vs telómero:  ρ=-0.759  (biología: más PDL → telómeros más cortos)
    PDL vs mtDNA:     ρ=+0.703  (biología: más PDL → más copias mtDNA)

  Correlaciones z_img vs biomarcadores (agregado por sample_id, n=77):
    telomere_length           | simple ρ=+0.107 (ns) | parcial ρ=-0.020 (ns, p=8.607e-01) | n=77
    mtdna_cn                  | simple ρ=+0.071 (ns) | parcial ρ=+0.225 (*, p=4.883e-02) | n=77
    PDL (sanity check)        | ρ=+0.183 (p=1.10e-01)


## SECCIÓN 8 — SERIE EXPERIMENTAL COMPLETA
Evidencia de que A2 es el mejor compromiso tras 7 variantes.

In [9]:
print("\n" + "=" * 70)
print("  SERIE EXPERIMENTAL COMPLETA — MVP-1")
print("=" * 70)

experiments = {
    "A0 baseline (10x+20x)": {
        "data": "763 imgs (10x+20x)", "losses": "MAE",
        "rho": 0.640, "worst": 0.442, "delta_sp": None,
        "mag_auc": 0.995, "mtdna": "ns",
        "finding": "Batch probe ~1.0 en todo. Shortcut learning confirmado."
    },
    "A0 baseline (10x)": {
        "data": "393 imgs (10x)", "losses": "MAE",
        "rho": 0.738, "worst": 0.616, "delta_sp": None,
        "mag_auc": None, "mtdna": "ns",
        "finding": "Quitar magnificación mejora todo. Fold 2 cruza 0.6."
    },
    "A1 rank (10x)": {
        "data": "393 imgs (10x)", "losses": "MAE + ranking (norma L2)",
        "rho": 0.754, "worst": 0.617, "delta_sp": 0.148,
        "mag_auc": None, "mtdna": "0.154 ns",
        "finding": "Ranking mejora robustez. Mejor ρ promedio."
    },
    ">>> A2 rank+cons (10x)": {
        "data": "393 imgs (10x)", "losses": "MAE + ranking (norma L2) + consistency",
        "rho": 0.736, "worst": 0.617, "delta_sp": 0.132,
        "mag_auc": None, "mtdna": "0.252 * (p=0.027)",
        "finding": "ÚNICA correlación parcial significativa con mtDNA. MODELO SELECCIONADO."
    },
    "A3 DANN (10x)": {
        "data": "393 imgs (10x)", "losses": "MAE + ranking + cons + DANN(sp)",
        "rho": 0.734, "worst": 0.595, "delta_sp": 0.125,
        "mag_auc": None, "mtdna": "0.193 ns",
        "finding": "DANN baja batch marginalmente pero worst fold < 0.6. Pierde mtDNA."
    },
    "A4 DANN cond (10x+20x)": {
        "data": "761 imgs (10x+20x)", "losses": "MAE + rank(escalar) + cons + DANN(mag) + DANN_cond(sp)",
        "rho": 0.638, "worst": 0.399, "delta_sp": 0.076,
        "mag_auc": 0.827, "mtdna": "0.185 ns",
        "finding": "ΔAUC pasa ≤0.10 pero fold 2 colapsa. DANN demasiado agresivo."
    },
    "Two-Stage frozen": {
        "data": "S1:2247 → S2:393 (10x)", "losses": "S1: MAE+rank+cons+DANN(mag) → S2: MAE",
        "rho": 0.648, "worst": 0.589, "delta_sp": 0.110,
        "mag_auc": None, "mtdna": "0.217 ns (p=0.058)",
        "finding": "Fold 2 mejora +0.19 vs A4. Pierde especificidad en folds 0/1."
    },
    "Two-Stage partial FT": {
        "data": "S1:2247 → S2:393 (10x)", "losses": "S1: idem → S2: MAE+rank, LR bajo",
        "rho": 0.649, "worst": 0.559, "delta_sp": 0.126,
        "mag_auc": None, "mtdna": "0.198 ns",
        "finding": "Fine-tune parcial empeora fold 2. Overfitting al dominio mayoritario."
    },
}

# Tabla resumen
print(f"\n  {'Modelo':30s} | {'ρ mean':>7s} | {'worst':>6s} | {'Δsp':>6s} | mtDNA parcial")
print(f"  {'-'*30}-+-{'-'*7}-+-{'-'*6}-+-{'-'*6}-+-{'-'*20}")
for name, e in experiments.items():
    d = f"{e['delta_sp']:+.3f}" if e['delta_sp'] is not None else "  n/a"
    marker = "◄◄◄" if ">>>" in name else ""
    print(f"  {name:30s} | {e['rho']:7.3f} | {e['worst']:6.3f} | {d} | {e['mtdna']} {marker}")

# Narrativa
print(f"\n  📋 Hallazgos clave de la serie experimental:")
print(f"  1. Magnificación era un confounder activo (AUC=0.995). Filtrarlo a 10x mejoró")
print(f"     worst fold de 0.442 → 0.616 sin cambiar el modelo.")
print(f"  2. Study_part está acoplado con biología (PDL+cell_line predicen sp con AUC=0.85).")
print(f"     DANN contra study_part inevitablemente borra señal biológica.")
print(f"  3. Ranking loss mejora robustez; consistency loss habilita fusion-readiness.")
print(f"  4. A2 es el único modelo con correlación parcial significativa con un biomarcador")
print(f"     molecular (mtDNA cn, ρ=0.252, p=0.027) controlando PDL y cell_line.")
print(f"  5. Más datos (Two-Stage) estabiliza folds pero pierde especificidad.")
print(f"     La fusión multimodal es la vía correcta para mejorar, no más regularización.")


  SERIE EXPERIMENTAL COMPLETA — MVP-1

  Modelo                         |  ρ mean |  worst |    Δsp | mtDNA parcial
  -------------------------------+---------+--------+--------+---------------------
  A0 baseline (10x+20x)          |   0.640 |  0.442 |   n/a | ns 
  A0 baseline (10x)              |   0.738 |  0.616 |   n/a | ns 
  A1 rank (10x)                  |   0.754 |  0.617 | +0.148 | 0.154 ns 
  >>> A2 rank+cons (10x)         |   0.736 |  0.617 | +0.132 | 0.252 * (p=0.027) ◄◄◄
  A3 DANN (10x)                  |   0.734 |  0.595 | +0.125 | 0.193 ns 
  A4 DANN cond (10x+20x)         |   0.638 |  0.399 | +0.076 | 0.185 ns 
  Two-Stage frozen               |   0.648 |  0.589 | +0.110 | 0.217 ns (p=0.058) 
  Two-Stage partial FT           |   0.649 |  0.559 | +0.126 | 0.198 ns 

  📋 Hallazgos clave de la serie experimental:
  1. Magnificación era un confounder activo (AUC=0.995). Filtrarlo a 10x mejoró
     worst fold de 0.442 → 0.616 sin cambiar el modelo.
  2. Study_part está aco

## SECCIÓN 9 — LIMITACIONES DOCUMENTADAS

In [10]:
print("\n" + "=" * 70)
print("  LIMITACIONES DOCUMENTADAS — MVP-1")
print("=" * 70)

limitations = [
    {
        "id": "L1",
        "title": "Contaminación residual de protocolo (study_part)",
        "detail": (
            f"El embedding z_img agrega ΔAUC={delta_sp:+.3f} de información sobre study_part "
            f"más allá de lo que PDL+cell_line explican. Parte de esto es artefacto de adquisición "
            f"(protocolo, microscopio, iluminación por fase) y parte es correlación biológica "
            f"legítima (fases tardías tienen PDL alto y morfología distinta). "
            f"No es posible separar ambas con los datos actuales."
        ),
        "mitigation": "La fusión multimodal con ómicas (que no comparten artefactos de imagen) "
                      "mitigará naturalmente esta limitación."
    },
    {
        "id": "L2",
        "title": "Fold 2 limitado (49 imágenes, 3 donantes, PhaseI/III)",
        "detail": (
            "El fold de validación más difícil contiene solo 49 imágenes de hFB1, hFB11 y hFB2, "
            "que provienen de PhaseI y PhaseIII. El train está dominado por PhaseII/V. "
            "Esto crea un domain shift natural que limita la generalización. "
            "hFB6/7/8 (originalmente asignados a folds) son SURF1 y se excluyeron correctamente."
        ),
        "mitigation": "Se demostró (Two-Stage) que preentrenar con más datos mejora fold 2 "
                      "(0.399→0.589). El encoder se beneficiaría de más donantes sanos en fases tempranas."
    },
    {
        "id": "L3",
        "title": "Fusion-readiness débil con telómero",
        "detail": (
            "La correlación parcial z_img vs telomere_length no es significativa en ninguna variante, "
            "a pesar de que PDL correlaciona fuertemente con telómero (ρ=-0.805). Esto sugiere que "
            "la señal morfológica capturable en brightfield 10x es más afín a procesos mitocondriales "
            "(mtDNA cn) que a dinámica telomérica."
        ),
        "mitigation": "El encoder de metilación (MVP-2) capturará señal telomérica directamente "
                      "vía relojes epigenéticos."
    },
    {
        "id": "L4",
        "title": "Dataset limitado a fibroblastos dérmicos in vitro",
        "detail": (
            "Todos los resultados son sobre fibroblastos primarios humanos en cultivo (Sturm et al.). "
            "No se ha validado en otros tipos celulares ni en tejido in vivo. "
            "La generalización a GSE113957/GSE130973 queda como trabajo futuro."
        ),
        "mitigation": "Planteado como validación externa en la tesis."
    },
]

for lim in limitations:
    print(f"\n  [{lim['id']}] {lim['title']}")
    print(f"      {lim['detail']}")
    print(f"      Mitigación: {lim['mitigation']}")


  LIMITACIONES DOCUMENTADAS — MVP-1

  [L1] Contaminación residual de protocolo (study_part)
      El embedding z_img agrega ΔAUC=+0.139 de información sobre study_part más allá de lo que PDL+cell_line explican. Parte de esto es artefacto de adquisición (protocolo, microscopio, iluminación por fase) y parte es correlación biológica legítima (fases tardías tienen PDL alto y morfología distinta). No es posible separar ambas con los datos actuales.
      Mitigación: La fusión multimodal con ómicas (que no comparten artefactos de imagen) mitigará naturalmente esta limitación.

  [L2] Fold 2 limitado (49 imágenes, 3 donantes, PhaseI/III)
      El fold de validación más difícil contiene solo 49 imágenes de hFB1, hFB11 y hFB2, que provienen de PhaseI y PhaseIII. El train está dominado por PhaseII/V. Esto crea un domain shift natural que limita la generalización. hFB6/7/8 (originalmente asignados a folds) son SURF1 y se excluyeron correctamente.
      Mitigación: Se demostró (Two-Stage) que p

## SECCIÓN 10 — EXPORTAR PARA MVP-2

In [11]:
print("\n" + "=" * 70)
print("  EXPORTAR MODELO Y EMBEDDINGS")
print("=" * 70)

# 1. Embeddings CSV
emb_path = os.path.join(OUTPUT_DIR, "A2_final_embeddings.csv")
df_embeddings.to_csv(emb_path, index=False)
print(f"  💾 Embeddings: {emb_path}")

# 2. Reporte JSON completo
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "A2: MAE + ranking(scalar) + consistency, ResNet-34 frozen, 10x only",
    "dataset": {
        "manifest": os.path.basename(MANIFEST_PATH),
        "filter": "Control, Normal, 10x only",
        "n_images": len(df),
        "n_cell_lines": len(cell_lines),
        "cell_lines": cell_lines,
        "n_folds": len(folds),
    },
    "hyperparams": {
        "batch_size": BATCH_SIZE, "epochs": EPOCHS, "patience": PATIENCE,
        "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY,
        "embedding_dim": EMBEDDING_DIM, "img_size": IMG_SIZE,
        "lambda_pdl": LAMBDA_PDL, "lambda_rank": LAMBDA_RANK,
        "lambda_cons": LAMBDA_CONS,
    },
    "results_per_fold": {str(k): v for k, v in results_per_fold.items()},
    "aggregated": {
        "mean_spearman": float(np.mean(spears)),
        "std_spearman": float(np.std(spears)),
        "worst_spearman": float(min(spears)),
        "mean_mae": float(np.mean(maes)),
        "std_mae": float(np.std(maes)),
        "mean_improvement_pct": float(np.mean(imps)),
    },
    "batch_probe": probe_results,
    "fusion_readiness": fusion_results,
    "pida_criteria": {
        "spearman_ge_0.6": bool(np.mean(spears) >= 0.6),
        "improvement_ge_25pct": bool(np.mean(imps) >= 25),
        "worst_fold_ge_0.6": bool(min(spears) >= 0.6),
    },
    "limitations": [{"id": l["id"], "title": l["title"]} for l in limitations],
    "experiments_summary": {
        name: {"rho": e["rho"], "worst": e["worst"], "delta_sp": e["delta_sp"],
               "mtdna": e["mtdna"], "finding": e["finding"]}
        for name, e in experiments.items()
    },
    "contrato_interfaz": {
        "input": "imagen brightfield 10x (224x224, RGB, ImageNet-normalized)",
        "output_embedding": f"torch.Tensor shape (1, {EMBEDDING_DIM})",
        "output_pdl": "torch.Tensor shape (1,), rango [0, 1]",
        "uso_para_fusion": "embedding → fusión PoE o concat+mask con encoders ómicos",
    },
}

report_path = os.path.join(OUTPUT_DIR, "mvp1_final_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"  💾 Reporte: {report_path}")

# 3. Lista de archivos
print(f"\n  📁 Archivos en {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    print(f"     {fname:40s} ({fsize/1024:.1f} KB)")

print(f"\n{'='*70}")
print(f"  MVP-1 CERRADO")
print(f"  Encoder: A2 (MAE + ranking + consistency, 10x)")
print(f"  Spearman: {np.mean(spears):.3f} ± {np.std(spears):.3f}")
print(f"  Worst fold: {min(spears):.3f}")
print(f"  mtDNA parcial: {fusion_results.get(COL_MTDNA, {}).get('partial_rho', 'n/a')}")
print(f"  Siguiente: MVP-2 (encoders ómicos)")
print(f"{'='*70}")


  EXPORTAR MODELO Y EMBEDDINGS
  💾 Embeddings: /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final/A2_final_embeddings.csv
  💾 Reporte: /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final/mvp1_final_report.json

  📁 Archivos en /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_final:
     A2_final_embeddings.csv                  (686.7 KB)
     A2_final_fold0.0.pt                      (83855.5 KB)
     A2_final_fold1.0.pt                      (83855.5 KB)
     A2_final_fold2.0.pt                      (83855.5 KB)
     mvp1_final_report.json                   (5.2 KB)

  MVP-1 CERRADO
  Encoder: A2 (MAE + ranking + consistency, 10x)
  Spearman: 0.733 ± 0.082
  Worst fold: 0.619
  mtDNA parcial: 0.22530101477469897
  Siguiente: MVP-2 (encoders ómicos)
